In [ ]:
"""Trains and evaluate a simple MLP
on the Reuters newswire topic classification task.
"""

import numpy as np
from tensorflow import keras
from tensorflow.keras.datasets import reuters
from tensorflow.keras.layers import Activation, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer

# The following import and function call are the only additions to code required
# to automatically log metrics and parameters to MLflow.
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Keras Example Comparison")
mlflow.tensorflow.autolog()

max_words = 1000
batch_size = 32
epochs = 5

print("Loading data...")
(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=max_words, test_split=0.2)

print(len(x_train), "train sequences")
print(len(x_test), "test sequences")

num_classes = int(np.max(y_train) + 1)
print(num_classes, "classes")

print("Vectorizing sequence data...")
tokenizer = Tokenizer(num_words=max_words)
x_train = tokenizer.sequences_to_matrix(x_train, mode="binary")
x_test = tokenizer.sequences_to_matrix(x_test, mode="binary")
print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)

print("Convert class vector to binary class matrix (for use with categorical_crossentropy)")
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("Building model...")
model = Sequential()
model.add(Dense(512, input_shape=(max_words,)))
model.add(Activation("relu"))
model.add(Dropout(0.5))
model.add(Dense(num_classes))
model.add(Activation("softmax"))

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
with mlflow.start_run(run_name="Keras Example Comparison Run"):
    history = model.fit(
        x_train, y_train, batch_size=batch_size, epochs=epochs, verbose=1, validation_split=0.1
    )
    score = model.evaluate(x_test, y_test, batch_size=batch_size, verbose=1)
    mlflow.log_metric("test_loss", score[0])
    mlflow.log_metric("test_accuracy", score[1])
    print("Test score:", score[0])
    print("Test accuracy:", score[1])

2026/06/08 01:39:33 INFO mlflow.tracking.fluent: Experiment with name 'Keras Example Comparison' does not exist. Creating a new experiment.


Loading data...
8982 train sequences
2246 test sequences
46 classes
Vectorizing sequence data...
x_train shape: (8982, 1000)
x_test shape: (2246, 1000)
Convert class vector to binary class matrix (for use with categorical_crossentropy)
y_train shape: (8982, 46)
y_test shape: (2246, 46)
Building model...


/Users/nrusnac/Dungeon/Internship/MLflow/ml/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
243/253 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5867 - loss: 1.8928

253/253 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6880 - loss: 1.4144 - val_accuracy: 0.7653 - val_loss: 1.0783
Epoch 2/5
252/253 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8051 - loss: 0.8461

253/253 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8178 - loss: 0.7806 - val_accuracy: 0.8009 - val_loss: 0.9011
Epoch 3/5
234/253 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8718 - loss: 0.5364

253/253 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8680 - loss: 0.5494 - val_accuracy: 0.8031 - val_loss: 0.8424
Epoch 4/5
253/253 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8976 - loss: 0.4135 - val_accuracy: 0.8087 - val_loss: 0.8732
Epoch 5/5
253/253 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9156 - loss: 0.3283 - val_accuracy: 0.8076 - val_loss: 0.8904
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step


2026/06/08 01:39:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 866us/step - accuracy: 0.7956 - loss: 0.8826
Test score: 0.8825593590736389
Test accuracy: 0.7956367135047913
🏃 View run Keras Example Comparison Run at: http://127.0.0.1:5000/#/experiments/5/runs/61a5d9d846f1492d8f01f807e7c494a6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


In [7]:
import json

with mlflow.start_run(run_name="Keras Example Comparison Run", run_id="61a5d9d846f1492d8f01f807e7c494a6"):
    example_texts = [
        "The stock market is doing well today.",
        "The new movie was fantastic!",
        "I am feeling sick and need to see a doctor."
    ]
    example_sequences = tokenizer.texts_to_sequences(example_texts)
    example_data = tokenizer.sequences_to_matrix(example_sequences, mode="binary")
    predictions = model.predict(example_data)
    predictions_list = predictions.tolist()
    artifacts = {
        "example_texts": example_texts,
        "predictions": predictions_list
    }
    with open("example_predictions.json", "w") as f:
        json.dump(artifacts, f)
    mlflow.log_artifact("example_predictions.json")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
🏃 View run Keras Example Comparison Run at: http://127.0.0.1:5000/#/experiments/5/runs/61a5d9d846f1492d8f01f807e7c494a6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


I registered and run_id as a model version 1, and add some description and tags using the MLflow UI

In [8]:
with mlflow.start_run(run_name="Keras Example Comparison Run", run_id="61a5d9d846f1492d8f01f807e7c494a6"):
    params = {
        "dense_1_units": 512,
        "dropout_rate": 0.5,
        "dense_1_activation": "relu",
        "output_activation": "softmax",
        "loss_function": "categorical_crossentropy",
        "max_words": max_words,
        "vectorization_mode": "binary",
        "num_classes": num_classes,
    }
    mlflow.log_params(params)

🏃 View run Keras Example Comparison Run at: http://127.0.0.1:5000/#/experiments/5/runs/61a5d9d846f1492d8f01f807e7c494a6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


Here is the improved a little model with the best test accuracy of 0.92, and I also logged the test loss and test accuracy as metrics in MLflow.

In [ ]:
"""Trains and evaluate a simple MLP
on the Reuters newswire topic classification task.
"""

import numpy as np
from tensorflow import keras
from tensorflow.keras.datasets import reuters
from tensorflow.keras.layers import Activation, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer

# The following import and function call are the only additions to code required
# to automatically log metrics and parameters to MLflow.
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Keras Example Comparison")
mlflow.tensorflow.autolog()

max_words = 1100 
batch_size = 32
epochs = 4 # from the results of the previous run, we can see after epoch 4 overfitting starts

print("Loading data...")
(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=max_words, test_split=0.2)

print(len(x_train), "train sequences")
print(len(x_test), "test sequences")

num_classes = int(np.max(y_train) + 1)
print(num_classes, "classes")

print("Vectorizing sequence data...")
tokenizer = Tokenizer(num_words=max_words)
x_train = tokenizer.sequences_to_matrix(x_train, mode="binary")
x_test = tokenizer.sequences_to_matrix(x_test, mode="binary")
print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)

print("Convert class vector to binary class matrix (for use with categorical_crossentropy)")
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("Building model...")
model = Sequential()
model.add(Dense(512*2, input_shape=(max_words,))) # increasing the number of units in the dense layer to 1024 to improve performance
model.add(Activation("relu"))
model.add(Dropout(0.5))
model.add(Dense(num_classes))
model.add(Activation("softmax"))

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
with mlflow.start_run(run_name="Keras Example Improve"):
    history = model.fit(
        x_train, y_train, batch_size=batch_size, epochs=epochs, verbose=1, validation_split=0.1
    )
    score = model.evaluate(x_test, y_test, batch_size=batch_size, verbose=1)
    mlflow.log_metric("test_loss", score[0])
    mlflow.log_metric("test_accuracy", score[1])
    print("Test score:", score[0])
    print("Test accuracy:", score[1]) # best acc 0.92

Loading data...
8982 train sequences
2246 test sequences
46 classes
Vectorizing sequence data...
x_train shape: (8982, 1100)
x_test shape: (2246, 1100)
Convert class vector to binary class matrix (for use with categorical_crossentropy)
y_train shape: (8982, 46)
y_test shape: (2246, 46)
Building model...
🏃 View run burly-fish-969 at: http://127.0.0.1:5000/#/experiments/5/runs/f8e2d603c0a3427ab35e7d92b58af193
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


Epoch 1/4
251/253 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3682 - loss: 2.8935

253/253 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4638 - loss: 2.3800 - val_accuracy: 0.5239 - val_loss: 1.9010
Epoch 2/4
230/253 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5493 - loss: 1.8350

253/253 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5650 - loss: 1.7631 - val_accuracy: 0.6129 - val_loss: 1.6956
Epoch 3/4
253/253 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6044 - loss: 1.6644

253/253 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6155 - loss: 1.6092 - val_accuracy: 0.6552 - val_loss: 1.5820
Epoch 4/4
233/253 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6545 - loss: 1.4840

253/253 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6550 - loss: 1.4885 - val_accuracy: 0.6874 - val_loss: 1.4989
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step


2026/06/08 18:04:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 977us/step - accuracy: 0.6679 - loss: 1.4808
Test score: 1.480781078338623
Test accuracy: 0.6678539514541626
🏃 View run Keras Example Improve 2 at: http://127.0.0.1:5000/#/experiments/5/runs/0d3930138733493791854b7cfb65474f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
